# Quad Lens with Time Delays — Direct Fit

## Learning to Autolens / Examples / quad_time_delay

---

**Problem.** A point-source quasar at $z_S = 2.0$ is quadruply imaged by a foreground galaxy at $z_L = 0.5$. The four images arrive at different times (a few weeks' delay between the leading and trailing images), and those delays carry direct information about the *time-delay distance* $D_{\Delta t}$ — and through it, the Hubble constant $H_0$.

**Method.** Two phases of the same lens model:

- **Phase 1 — Lens recovery (cosmology fixed).** Fit the PowerLaw + ExternalShear lens and `ps.Point` source position with $H_0 = 70$ pinned. Recover image positions, time delays, and magnifications.
- **Phase 2 — $H_0$ recovery (free).** Same model, but free `cosmology.H0` with prior `Uniform(40, 120)`. The headline TDCOSMO result: how well does a single quad with HST positions + 0.5-day delays constrain $H_0$?

**Why this matters.** Strong-lens time delays are one of three ladder-independent routes to $H_0$, alongside SH0ES (Cepheid + Type Ia) and Planck CMB. The H0LiCOW / TDCOSMO collaborations have published $H_0 = 73.3^{+1.7}_{-1.8}$ km/s/Mpc from 6 well-modeled time-delay quads (Wong et al. 2020). The mass-sheet degeneracy (MSD) is the dominant systematic and is not addressed here — see Exercise 3.

**Prerequisites.** Mod 03 (Nautilus + priors), Mod 11 (audit methodology). The point-source likelihood is *new* to this notebook compared to everything else in the curriculum.

**Key references.**
- Refsdal (1964), MNRAS 128, 307 — original time-delay → $H_0$ proposal.
- Wong+ (2020), MNRAS 498, 1420 — H0LiCOW final 2.4% measurement.
- Birrer+ (2020), A&A 643, A165 — TDCOSMO 2020, MSD-aware re-analysis.
- `autolens_workspace_latest/scripts/point_source/features/time_delays.py`.

In [ ]:
import os
if os.environ.get("PYAUTOFIT_TEST_MODE"):
    raise RuntimeError(
        f"PYAUTOFIT_TEST_MODE={os.environ['PYAUTOFIT_TEST_MODE']!r} is set. "
        "Unset it and restart the kernel: unset PYAUTOFIT_TEST_MODE"
    )

import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, IFrame, Markdown, display

import autofit as af
import autolens as al
import autolens.plot as aplt

%matplotlib inline
print(f"PyAutoLens version: {al.__version__}")

In [ ]:
RESULTS_ROOT = Path("results")

def show_result(stage_name):
    stage_dir = RESULTS_ROOT / stage_name
    if not stage_dir.exists():
        print(f"(no results for stage {stage_name!r} yet — run on Cannon or set LTA_RUN_HEAVY=1)")
        return

    summary_path = stage_dir / "summary.json"
    if summary_path.exists():
        s = json.loads(summary_path.read_text())
        bullets = [f"### `{stage_name}`"]
        if s.get("max_log_likelihood") is not None:
            bullets.append(f"- max log-likelihood: **{s['max_log_likelihood']:.2f}**")
        if s.get("log_evidence") is not None:
            bullets.append(f"- log evidence: **{s['log_evidence']:.2f}**")
        display(Markdown("\n".join(bullets)))

    for ext, label in (("fit_subplot.png", "Fit subplot"),
                       ("corner.pdf", "Corner plot")):
        p = stage_dir / ext
        if p.exists():
            display(Markdown(f"**{label}:**"))
            if ext.endswith(".pdf"):
                display(IFrame(src=str(p), width=720, height=720))
            else:
                display(Image(filename=str(p)))

    mr = stage_dir / "model_results.txt"
    if mr.exists():
        preview = "\n".join(mr.read_text().splitlines()[:50])
        display(Markdown(f"**`model_results.txt`** (first 50 lines):\n\n```\n{preview}\n```"))

---

## 1. Load the dataset

Unlike the imaging examples, point-source datasets are JSON-serialised `al.PointDataset` objects: a small list of image positions, optional fluxes, and time delays plus their uncertainties. There's no FITS file to mask or PSF to convolve — we have a list of $(y, x)$ tuples and a list of arrival times.

In [ ]:
dataset_path = Path("mocks")
dataset = al.from_json(file_path=dataset_path / "point_dataset.json")
truths = json.loads((dataset_path / "truths.json").read_text())

print(f"Dataset name: {dataset.name}")
print(f"Number of images: {len(dataset.positions)}")
print()
for i, (pos, td, flux) in enumerate(zip(
    np.asarray(dataset.positions),
    np.asarray(dataset.time_delays),
    np.asarray(dataset.fluxes),
)):
    print(f"  image {i}:  pos = ({pos[0]:+.4f}, {pos[1]:+.4f})\"   "
          f"t = {td:+8.3f} d   flux = {flux:6.3f}")

print()
print("Position noise:    {:.4f} arcsec".format(np.asarray(dataset.positions_noise_map).mean()))
print("Time-delay noise:  {:.2f} days".format(np.asarray(dataset.time_delays_noise_map).mean()))
print("Flux noise (frac): {:.3f}".format(np.median(np.asarray(dataset.fluxes_noise_map) /
                                                       np.abs(np.asarray(dataset.fluxes)))))

## 2. Inspect — image configuration, time delays, magnifications

Plot the four images on a sky-plane axis with relative time delays annotated. With these four images visible, we can already estimate the Einstein radius geometrically — the four images sit on the lens's tangential critical curve, and their separation gives a rough $\theta_E \sim 1\,\mathrm{arcsec}$ at this geometry.

Time delays are normalised to the **leading image** (smallest $t$). The image with the most negative arrival time *got there first*; larger values are more delayed.

In [ ]:
positions = np.asarray(dataset.positions)
time_delays = np.asarray(dataset.time_delays)
fluxes = np.asarray(dataset.fluxes)

# Relative delays (image with smallest t is the reference)
leader_idx = int(np.argmin(time_delays))
delays_rel = time_delays - time_delays[leader_idx]

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(positions[:, 1], positions[:, 0],
           c=fluxes, s=200, cmap="viridis", edgecolor="k", zorder=3)
ax.scatter([0], [0], marker="x", color="red", s=200, zorder=4,
           label=f"lens centre (z={truths['redshifts']['z_lens']})")
ax.scatter([truths['source']['centre'][1]], [truths['source']['centre'][0]],
           marker="*", color="orange", s=400, zorder=4,
           label=f"source (z={truths['redshifts']['z_source']}, true position)")
for i, (p, dt) in enumerate(zip(positions, delays_rel)):
    label = f"  img{i}\n  Δt={dt:+.2f}d"
    ax.annotate(label, xy=(p[1], p[0]), xytext=(p[1]+0.05, p[0]+0.05),
                fontsize=9, color="black")
circle = plt.Circle((0, 0), truths['mass']['einstein_radius'],
                    fill=False, ls="--", color="grey",
                    label=f"true θ_E = {truths['mass']['einstein_radius']}\"")
ax.add_patch(circle)
ax.set_xlabel("x (arcsec)"); ax.set_ylabel("y (arcsec)")
ax.set_xlim(-1.7, 1.7); ax.set_ylim(-1.7, 1.7)
ax.set_aspect("equal"); ax.legend(loc="upper right", fontsize=9)
ax.set_title("Quad image configuration with time delays (relative to leader)")
plt.tight_layout()
plt.show()

print(f"\nLeader image: img{leader_idx}")
print(f"Delay span: 0 → {delays_rel.max():.2f} days")
print(f"Magnifications (truth): {truths['truth_magnifications']}")

---

## 3. Phase 1 model — lens parameters with cosmology fixed

**Mass model.** PowerLaw + ExternalShear (the TDCOSMO standard). Letting the slope $\gamma'$ float matters here in a way it didn't for the imaging examples — for time-delay cosmography the mass profile shape couples directly to $D_{\Delta t}$, so a steeper or shallower slope changes the inferred $H_0$. Even with cosmology fixed in Phase 1, we let $\gamma'$ float to recover the truth.

**Source.** `al.ps.Point` — just a 2-parameter centre. The intrinsic flux is fit through magnification (one global flux scale).

**Cosmology.** `al.cosmo.FlatLambdaCDM(H0=70, Om0=0.30)` — fixed. Phase 2 will free $H_0$.

**Priors.** Loose enough to explore, tight enough to avoid the rotational mirror-image (Pattern A) failure mode. The lens centre prior is the tightest (`σ=0.1\"`) since for a real target this would be set from imaging of the lens galaxy.

In [ ]:
mass = af.Model(al.mp.PowerLaw)
mass.centre.centre_0 = af.GaussianPrior(mean=0.0, sigma=0.1)
mass.centre.centre_1 = af.GaussianPrior(mean=0.0, sigma=0.1)
mass.einstein_radius = af.UniformPrior(lower_limit=0.5, upper_limit=2.5)
mass.slope = af.TruncatedGaussianPrior(mean=2.0, sigma=0.2,
                                       lower_limit=1.5, upper_limit=2.5)
mass.ell_comps.ell_comps_0 = af.TruncatedGaussianPrior(
    mean=0.0, sigma=0.3, lower_limit=-1.0, upper_limit=1.0)
mass.ell_comps.ell_comps_1 = af.TruncatedGaussianPrior(
    mean=0.0, sigma=0.3, lower_limit=-1.0, upper_limit=1.0)

shear = af.Model(al.mp.ExternalShear)
shear.gamma_1 = af.GaussianPrior(mean=0.0, sigma=0.1)
shear.gamma_2 = af.GaussianPrior(mean=0.0, sigma=0.1)

lens = af.Model(al.Galaxy, redshift=0.5, mass=mass, shear=shear)

point_0 = af.Model(al.ps.Point)
point_0.centre.centre_0 = af.GaussianPrior(mean=0.0, sigma=0.3)
point_0.centre.centre_1 = af.GaussianPrior(mean=0.0, sigma=0.3)
source = af.Model(al.Galaxy, redshift=2.0, point_0=point_0)

model_phase_1 = af.Collection(galaxies=af.Collection(lens=lens, source=source))
print(f"Phase 1 free parameters: {model_phase_1.prior_count}")

---

## 4. Phase 1 fit (skip-guard pattern)

Point-source fits are dramatically cheaper than imaging fits — at $\sim 9$ free parameters and $n_\mathrm{live}=100$, this converges in 10–30 minutes on 32 cores. Still gated by `LTA_RUN_HEAVY=1` for consistency with the other examples; the committed Cannon results are loaded by default.

In [ ]:
_HAVE_P1 = (RESULTS_ROOT / "quad_direct_fit" / "summary.json").exists()
_FORCE = os.environ.get("LTA_RUN_HEAVY", "").lower() in ("1", "true", "yes")

if _HAVE_P1 and not _FORCE:
    print("[Phase 1] Loading committed Cannon result.")
    show_result("quad_direct_fit")
    result_phase_1 = None
elif not _HAVE_P1 and not _FORCE:
    print("[Phase 1] No committed results yet.")
    print("[Phase 1] Submit on Cannon via fit_example_quad_time_delay.py --part direct")
    print("[Phase 1] or set LTA_RUN_HEAVY=1 to run locally (~10-30 min).")
    result_phase_1 = None
else:
    cosmology = al.cosmo.FlatLambdaCDM(H0=70.0, Om0=0.30)
    grid = al.Grid2D.uniform(shape_native=(150, 150), pixel_scales=0.04)
    solver = al.PointSolver.for_grid(
        grid=grid, pixel_scale_precision=0.001, magnification_threshold=0.1,
    )
    analysis = al.AnalysisPoint(
        dataset=dataset, solver=solver, cosmology=cosmology, use_jax=False,
    )
    search = af.Nautilus(
        path_prefix=Path("output") / "quad_time_delay",
        name="quad_direct_fit",
        unique_tag="phase_1_cosmology_fixed",
        n_live=100, n_batch=50, iterations_per_update=5000,
        number_of_cores=int(os.environ.get("SLURM_CPUS_PER_TASK", "1")),
    )
    result_phase_1 = search.fit(model=model_phase_1, analysis=analysis)
    print("[Phase 1] Done.")
    print(result_phase_1.info)

---

## 5. Phase 2 — free $H_0$

Same model, plus `cosmology = af.Model(al.cosmo.FlatLambdaCDM)` with `H0` as a free parameter (`Uniform(40, 120)`) and `Om0` fixed to 0.30. We don't reuse Phase 1's posterior as a prior here — point-source fits are fast, and we want the joint H0–lens posterior to capture the $H_0$-vs-$\theta_E$ degeneracy correctly.

$H_0$ recovery scales with the precision of the **time-delay** measurement (positions only constrain the *ratios* of distances, not the absolute scale). With a 0.5-day uncertainty on a ~21-day delay span (2.4% relative), expect $H_0$ to be recovered to a few percent — Wong+ 2020's H0LiCOW achieved 2.4% on the *combined* 6 quads.

In [ ]:
# Re-build the same lens/source model, then attach a free-H0 cosmology.
mass = af.Model(al.mp.PowerLaw)
mass.centre.centre_0 = af.GaussianPrior(mean=0.0, sigma=0.1)
mass.centre.centre_1 = af.GaussianPrior(mean=0.0, sigma=0.1)
mass.einstein_radius = af.UniformPrior(lower_limit=0.5, upper_limit=2.5)
mass.slope = af.TruncatedGaussianPrior(mean=2.0, sigma=0.2,
                                       lower_limit=1.5, upper_limit=2.5)
mass.ell_comps.ell_comps_0 = af.TruncatedGaussianPrior(
    mean=0.0, sigma=0.3, lower_limit=-1.0, upper_limit=1.0)
mass.ell_comps.ell_comps_1 = af.TruncatedGaussianPrior(
    mean=0.0, sigma=0.3, lower_limit=-1.0, upper_limit=1.0)
shear = af.Model(al.mp.ExternalShear)
shear.gamma_1 = af.GaussianPrior(mean=0.0, sigma=0.1)
shear.gamma_2 = af.GaussianPrior(mean=0.0, sigma=0.1)
lens = af.Model(al.Galaxy, redshift=0.5, mass=mass, shear=shear)

point_0 = af.Model(al.ps.Point)
point_0.centre.centre_0 = af.GaussianPrior(mean=0.0, sigma=0.3)
point_0.centre.centre_1 = af.GaussianPrior(mean=0.0, sigma=0.3)
source = af.Model(al.Galaxy, redshift=2.0, point_0=point_0)

cosmology = af.Model(al.cosmo.FlatLambdaCDM)
cosmology.H0 = af.UniformPrior(lower_limit=40.0, upper_limit=120.0)
cosmology.Om0 = 0.30  # fix matter density; H0 is the cosmographic parameter

model_phase_2 = af.Collection(
    galaxies=af.Collection(lens=lens, source=source),
    cosmology=cosmology,
)
print(f"Phase 2 free parameters: {model_phase_2.prior_count}")

In [ ]:
_HAVE_P2 = (RESULTS_ROOT / "quad_direct_fit_h0_free" / "summary.json").exists()

if _HAVE_P2 and not _FORCE:
    print("[Phase 2] Loading committed Cannon result.")
    show_result("quad_direct_fit_h0_free")
    result_phase_2 = None
elif not _HAVE_P2 and not _FORCE:
    print("[Phase 2] No committed results yet.")
    print("[Phase 2] Submit via fit_example_quad_time_delay.py --part direct_h0_free")
    print("[Phase 2] or set LTA_RUN_HEAVY=1 to run locally (~10-30 min).")
    result_phase_2 = None
else:
    grid = al.Grid2D.uniform(shape_native=(150, 150), pixel_scales=0.04)
    solver = al.PointSolver.for_grid(
        grid=grid, pixel_scale_precision=0.001, magnification_threshold=0.1,
    )
    analysis = al.AnalysisPoint(dataset=dataset, solver=solver, use_jax=False)
    search = af.Nautilus(
        path_prefix=Path("output") / "quad_time_delay",
        name="quad_direct_fit",
        unique_tag="phase_2_h0_free",
        n_live=150, n_batch=50, iterations_per_update=5000,
        number_of_cores=int(os.environ.get("SLURM_CPUS_PER_TASK", "1")),
    )
    result_phase_2 = search.fit(model=model_phase_2, analysis=analysis)
    print("[Phase 2] Done.")
    print(result_phase_2.info)

---

## 6. Audit (point-source flavour)

The `/autolens-fit-diagnostics` skill is calibrated for *imaging* fits — `chi²/N`, `max|res|σ`, residual maps. Point-source fits don't have a residual *map*; instead we have:

- **Position residuals**: model image positions vs. data positions, in arcseconds. Bar: $\lesssim 1\sigma_\mathrm{pos}$ across all images is PASS, $\gtrsim 3\sigma_\mathrm{pos}$ is FAIL.
- **Time-delay residuals**: model delays vs. data delays. Bar: same.
- **Magnification ratios**: only meaningful if `fluxes` are fit (we did include them). A factor-of-2 mismatch suggests microlensing or an unmodelled lens-light contaminant.
- **Posterior on $\gamma'$**: should be tight around 2.0 since the truth is isothermal. A pinned-at-rail (1.5 or 2.5) would indicate the prior is fighting the data.
- **Posterior on $H_0$ (Phase 2)**: should bracket the truth (70 km/s/Mpc) within $\sim 5\%$. A skewed posterior off the truth would flag the mass-sheet degeneracy or an unrecovered shear mode.

### Verdict template

```
Phase 1 (cosmology fixed)
  Verdict:          [ PASS | SUSPECT | FAIL ]
  Position resids:  ___σ_pos max
  Delay resids:     ___σ_t max
  γ' recovered:     _.___ ± _.___    (truth: 2.000)
  θ_E recovered:    _.___ ± _.___    (truth: 1.200)
  log_evidence:     ____

Phase 2 (H0 free)
  Verdict:          [ PASS | SUSPECT | FAIL ]
  H0 recovered:     ___.__ ± _.__ km/s/Mpc    (truth: 70.0)
  H0 fractional σ:  __.__ %
  γ'-H0 covariance: ____ (visible in corner.pdf?)
  log_evidence:     ____    (vs Phase 1)
```

*Worked verdict pending Cannon results.*

---

## 7. Exercises

### Exercise 1 — Positions-only baseline
Re-run Phase 2 with `time_delays` zeroed out in the dataset (or use `al.PointDataset` constructed without delays). How well-constrained is $H_0$? You should see the posterior collapse to the prior — positions alone don't carry distance information. This is the *control* that proves time delays are doing the cosmographic work.

### Exercise 2 — Slope effect on H₀ (the mass-sheet degeneracy in disguise)
Repeat Phase 2 but **fix `slope=2.0`** (Isothermal). Compare $H_0$'s posterior to the slope-free version. The slope-fixed result has tighter $H_0$ uncertainty *and may be biased* — you've thrown away the parameter that absorbs MSD. This is exactly why H0LiCOW's $H_0$ uncertainty was driven by mass-profile assumptions.

### Exercise 3 — Add external convergence κ_ext
Construct a model with an additional `al.mp.MassSheet(kappa=...)` at the lens redshift, with a `Uniform(-0.1, +0.1)` prior on `kappa`. Re-fit Phase 2. $H_0$'s posterior should *inflate* by $\sim 5{-}10\%$. This is the **mass-sheet degeneracy** in its most explicit form: a sheet of uniform convergence preserves all image positions and rescales delays, so $H_0$ and $\kappa_\mathrm{ext}$ are formally degenerate. Breaking this requires stellar kinematics (TDCOSMO 2020, Birrer+).

### Exercise 4 — Shear marginalisation
Compare three setups: (a) `gamma_1, gamma_2` free with `Gaussian(0, 0.1)`; (b) fixed at truth; (c) frozen at zero. How much does $H_0$ shift between them? In a real measurement, line-of-sight shear from large-scale structure has $\sim 0.02$ amplitude — failing to marginalise over it leaves a systematic bias.

### Exercise 5 — Positions-noise scaling
Halve the position noise (`positions_noise_map → 0.0025\"`, JWST-class) and re-fit. Compare $H_0$ uncertainty. Then halve the time-delay noise (`0.25 d`). Which improvement is more impactful? (Spoiler: time-delay noise dominates.) This is the rationale for LSST + Roman synergy in the next decade — both sub-arcsec positions and dense delay sampling on 100s of quads.

---

## Summary

- The point-source likelihood (`al.AnalysisPoint`) is **dramatically lower-dimensional** than imaging — 9 to 11 free parameters here vs 30+ for an extended-arc fit. Nautilus converges in 10–30 minutes instead of hours.
- Time delays + positions + fluxes give cosmographic leverage that imaging alone cannot. With $H_0$ free in Phase 2, this single quad recovers the truth to a few percent.
- The mass-sheet degeneracy is the dominant systematic at the few-percent level. The exercises explore it explicitly — without external constraints (kinematics, kappa_ext priors), $H_0$ from a single quad is biased by $\sim 10\%$ in the worst case.
- This is a deliberately *clean* mock: a smooth PowerLaw + shear, no microlensing, no LOS structure. Real H0LiCOW / TDCOSMO targets each carry a 100-page systematics paper. The notebook here teaches the *anatomy* of the measurement; for the systematics, see Wong+ 2020 and Birrer+ 2020.

**Next steps:** there's no `02_quad_slam.ipynb` in this collection — point-source fits are 1-stage by nature, not 5-stage. The closest analog of 02 would be a *kinematics-augmented* fit, which belongs in a future module on TDCOSMO-class analyses.

---

*Learning to Autolens — Examples / quad_time_delay / 01*  
*Rodrigo Córdova Rosado, Harvard CfA*